In [34]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import concurrent.futures

import matplotlib.pyplot as plt
import matplotlib.patches as patches
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import plotly.graph_objects as go

from rod.helix import Helix
from rod.helix_util import HelixUtil

from energies.bend import Bend
from energies.bend_twist import BendTwist
from energies.gravity import Gravity
from energies.random import RandomForce
from energies.twist import Twist
from math_util.rotation import RotationUtil, Quaternion
from math_util.vectors import Vector
from rod.RodHelixConverter import RodHelixConverter
from rod.helix import Helix
from rod.helix_util import HelixUtil
from rod.preprocess import Preprocess
from rod.rod_generator import RodGenerator
from rod.rod_util import RodUtil
from solver.sim import Sim
from visualization.visualizer import Visualizer

In [98]:
# Methods to propagate the centerline and material frames of a helix

from visualization.visualizer import Visualizer

def strands_to_one_objs(strands: np.ndarray, frame_idx: int, output_file: str = None, y_up: bool = True):
    output_file = f"output/obj/obj_{frame_idx}.obj" if output_file is None else output_file
    Visualizer.clear_output_file(output_file)
    vertex_offset = 1
    for strand in strands:
        print("hi")
        pos = strand[:, :3]
        vertex_offset = Visualizer.to_simple_obj(pos=pos, output_file=output_file, init_offset=vertex_offset, y_up=y_up)
    return

def propagate_q(q: np.ndarray, r0: np.ndarray, n0: np.ndarray, n_sites: int, s: np.ndarray, r: np.ndarray,
                    n: np.ndarray):
        # Starting with the clamped material frame, integrate forward
        r[0, :] = r0
        n[0, :] = n0
        for i in range(1, n_sites):
            # Left hand side of interval (previous element)
            r_L = r[i - 1]
            n_L = n[i - 1]
            s_R, s_L = s[i], s[i - 1]
            s_sL = s_R - s_L
            # Twist and curvature
            tau, k_1, k_2 = q[3 * i - 3:3 * i]
            # Darboux vector and unit vector aligned with the Darboux vector
            Omega = tau * n_L[0, :] + k_1 * n_L[1, :] + k_2 * n_L[2, :]
            Omega_norm = np.linalg.norm(Omega)

            # Degenerate case: straight line/no change in material frame
            if Omega_norm < 1e-12:
                n[i] = n_L
                r[i] = r_L + n_L[0] * s_sL
                continue

            w = Omega / Omega_norm

            # Projection of vector parallel to and perpendicular to w
            n_L_par = np.dot(n_L, w)[:, np.newaxis] * w
            n_L_perp = n_L - n_L_par

            # Compute the material frame
            n_i = n_L_par + n_L_perp * np.cos(Omega_norm * s_sL) + np.cross(w, n_L_perp) * np.sin(Omega_norm * s_sL)
            n[i] = n_i

            # Compute the centerline
            n_0_parallel = n_L_par[0]
            n_0_perp = n_L_perp[0]
            r_i = (r_L + n_0_parallel * s_sL + n_0_perp * np.sin(Omega_norm * s_sL) / Omega_norm +
                   np.cross(w, n_0_perp) * (1 - np.cos(Omega_norm * s_sL)) / Omega_norm)
            r[i] = r_i
        return r, n

def propagate(helix: Helix) -> tuple[np.ndarray, np.ndarray]:
    """
    Computes the centerline and the material frames from the generalized coordinates [q]

    n_i(s) = n_{i, L}^{Q ||} + n_{i, L}^{Q perp} cos(Omega(s - s_L^Q)) + omega \cross n_{i, L}^{Q perp} sin(Omega(s - s_L^Q))
    """
    # Centerline and material frames
    r = np.zeros((helix.n_sites, 3))
    n = np.zeros((helix.n_sites, 3, 3))
    HelixUtil.propagate_q(helix.q, helix.r0, helix.n0, helix.n_sites, helix.s, r, n)
    return r, n

def plot_helix_plotly(helices, show_frames=True, frame_step=2, frame_scale=0.02, dot_pts=None):
    """
    Creates an interactive visualization of multiple helices' centerlines and material frames.

    Parameters:
    - helices: Array-like of tuples (r, n), where:
      - r: (n_sites, 3) Centerline positions
      - n: (n_sites, 3, 3) Material frames
    - show_frames: Whether to show material frames
    - frame_step: Step size for plotting frames (to avoid clutter)
    - frame_scale: Scaling factor for material frame vectors
    - dot_pts: List of indices at which to place red dots
    """
    # Create figure
    fig = go.Figure()

    # Plot each helix
    for idx, (r, n) in enumerate(helices):
        # Plot centerline
        fig.add_trace(go.Scatter3d(
            x=r[:, 0], y=r[:, 1], z=r[:, 2],
            mode='lines',
            line=dict(width=3),
            name=f'Centerline {idx + 1}'
        ))
    
        if dot_pts is not None:
                dot_pts = [i for i in dot_pts if 0 <= i < len(r)]  # sanitize
                fig.add_trace(go.Scatter3d(
                    x=[r[i, 0] for i in dot_pts],
                    y=[r[i, 1] for i in dot_pts],
                    z=[r[i, 2] for i in dot_pts],
                    mode='markers',
                    marker=dict(size=5, color='red'),
                    name='Marked Points'
                ))

        # Plot material frames as arrows
        if show_frames:
            for i in range(0, len(r), frame_step):
                origin = r[i]
                for j, color in enumerate(['red', 'green', 'black']):  # x (red), y (green), z (black)
                    frame_vec = n[i, j] * frame_scale  # Scale frame vectors
                    fig.add_trace(go.Scatter3d(
                        x=[origin[0], origin[0] + frame_vec[0]],
                        y=[origin[1], origin[1] + frame_vec[1]],
                        z=[origin[2], origin[2] + frame_vec[2]],
                        mode='lines',
                        line=dict(color=color, width=2),
                        showlegend=False  # Avoid excessive legend entries
                    ))

    # Update layout
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', range=[-10, 20]),
            yaxis=dict(title='Y', range=[-10, 20]),
            zaxis=dict(title='Z', range=[10, 40]),
            aspectmode='manual',
            aspectratio=dict(x=1, y=1, z=1)
        ),
        title=dict(text='Helix Visualization', y=0.95, x=0.5, xanchor='center', yanchor='top'),
    )

    fig.show()

### Switchbacks

In [113]:
n_sites = 100  # Number of sites along the helix
L = 10.0  # Total length
s = np.linspace(0, L, n_sites)  # Arc length array

# Initial position: Start the curl at the origin
r0 = np.array([0.0, 0.0, 0.0])

# Initial material frame: Oriented along the curl
pos, theta = RodGenerator.example_rod(n=100, curl_radius=1.5, curl_frequency=1, height_scale=0.3)
helix = RodHelixConverter.rod_to_helix(pos, theta)
# Create a twisting and bending motion in q
# twist = np.full(n_sites, np.pi / 4)
# # twist = np.zeros(n_sites)
# # curvature_x = 3.0 * np.cos(s * np.pi) # Varies to form a curl
# # curvature_y = 0 #3.0 * np.cos(s * np.pi) #+ np.random.random_sample()  # Varies to form a curl
# curvature_x = np.full(n_sites, 3.0)
# curvature_y = np.full(n_sites, 3.0)

# n_switchbacks = 3
# kappa_mag = 20.0  # Controls tightness of curls
# twist_mag = np.pi / 2  # Optional, controls 3D spiral effect

# Smoothed alternating curvature: cosine gives smooth flip in direction
# curvature_x = kappa_mag * np.cos(2 * np.pi * n_switchbacks * s)
# curvature_y = np.zeros_like(s)

# Optional: add twist that increases or oscillates with arc length
# twist = twist_mag * np.sin(2 * np.pi * n_switchbacks * s)  # Twist "switches back" too
# twist = 0
# Or keep constant for uniform helical torsion
# twist = np.full(n_sites, twist_mag)

# q = np.zeros((3 * n_sites,))
# q[0::3] = twist
# q[1::3] = curvature_x
# q[2::3] = curvature_y

# Define stiffness (optional)
EI = np.ones((3 * n_sites,)) * 0.1  # Uniform stiffness

# Create helix instance
# r0 and n0 are the position and material frame of the clamped top node
# helix_instance = Helix(q=q, q0=q, n_sites=n_sites, s=s, L=L, r0=r0, n0=n0, EI=EI)

#adding twists: indicies 0, 3, 6, 9, 12, 15, 18, 21, 24, 27
#helix.q[] = np.pi/2
helix.q[64] = np.pi #k1 
helix.q[70] = -np.pi #k1
# helix.q[75] = -np.pi/2


# Perform one propagation step
r, n = HelixUtil.propagate(helix)
strands = []
pos, _ = RodHelixConverter.helix_to_rod(helix)
strands.append(pos)

plot_helix_plotly([(r, n)], frame_scale=0.2, show_frames=False, dot_pts = [21, 23])
# strands_to_one_objs(np.array(strands), frame_idx=1)


### START HERE!
Implementing switchback insertion from here: https://www.cs.yale.edu/homes/wu-haomiao/publication/static/pdfs/main.pdf

In [125]:
# inserting a switchback centered at pt
'''
Input: helical solution, d (material frame)
'''
import numpy as np
from scipy.special import erf
from scipy.optimize import minimize
from copy import deepcopy

# variables
helical_r, helical_d = propagate(helix)
poisson_ratio = 0.5
big_gamma = 1 / (1 + poisson_ratio)

def switchback_k(helix, params, point):
    """
    Computes kappa_3^0(s) with switchback insertion.

    Parameters:
    - helix: Helix object with properties q, s
    - params: dict with keys ['a1', 'b1', 'c1', 'a2', 'b2', 'c2', 'a3', 'b3', 'c3', 'a4', 'c4']
    - point: int, index of the point where the switchback is inserted

    Returns:
    - k3_switch: np.ndarray of same shape as s
    """

    kappa_3_h = helix.q[2::3] 
    kappa_2_h = helix.q[1::3]
    kappa_1_h = helix.q[0::3]

    # per point - add 0 for first point
    s = helix.s # centered at top point
    s1 = -(s[:point] - s[point]) # upstream from switchback
    s2 = s[point:-1] - s[point] # downstream from switchback
    s_new = np.concatenate((s1, s2)) # np.ndarray of arc-length values (centered around point)
    s_new = np.insert(s_new, 0, 0)

    a1, b1, c1 = params['a1'], params['b1'], params['c1']
    a2, b2, c2 = params['a2'], params['b2'], params['c2']
    a3, b3, c3 = params['a3'], params['b3'], params['c3']
    b4, c4     = params['b4'], params['c4']

    a4_prefactor = np.sqrt(b4) * np.exp(c4**2 / (4 * b4))
    a4_term1 = 2*big_gamma*kappa_3_h / np.sqrt(np.pi)
    a4_term2 = a3/np.sqrt(b3) * np.exp(-c3**2 / (4 * b3))
    a4 = a4_prefactor * (a4_term1 - a4_term2)

    alpha = (a3 / 2*big_gamma) * np.sqrt(np.pi / b3) * np.exp(-c3**2 / (4 * b3))

    # k switchbacks
    k1_switch = a1 * np.exp(-b1 * s_new**2) * np.cos(c1 * s_new) + a2 * np.exp(-b2 * s_new**2) * np.cos(c2 * s_new) + kappa_1_h
    k2_switch = a3 * np.exp(-b3 * s_new**2) * np.cos(c3 * s_new) + a4 * np.exp(-b4 * s_new**2) * np.cos(c4 * s_new) + kappa_2_h
    
    k3_term1 = np.real(erf(np.sqrt(b3 * s_new) + (1j * c3 / (2 * np.sqrt(b3 * s_new)))))
    k3_term2 = np.real(erf(np.sqrt(b4 * s_new) + (1j * c4 / (2 * np.sqrt(b4 * s_new)))))

    k3_switch = -alpha * k3_term1 - (kappa_3_h - alpha) * k3_term2

    helix.q[0::3] = k1_switch
    helix.q[1::3] = k2_switch
    helix.q[2::3] = k3_switch

    return helix

def params_to_vec(params):
    keys = ['a1', 'b1', 'c1', 'a2', 'b2', 'c2', 'a3', 'b3', 'c3', 'b4', 'c4']
    return np.array([params[k] for k in keys])

def vec_to_params(vec):
    keys = ['a1', 'b1', 'c1', 'a2', 'b2', 'c2', 'a3', 'b3', 'c3', 'b4', 'c4']
    return dict(zip(keys, vec))

init_params = {
    'a1': 1.0, 'b1': 1.0, 'c1': 0.0,
    'a2': 0.8, 'b2': 1.0, 'c2': 0.0,
    'a3': 0.6, 'b3': 1.0, 'c3': 0.0,
    'b4': 1.0, 'c4': 0.0
}
vec0 = params_to_vec(init_params)
point = 21

def minimize_energy(params, helix, point):
    # Boundary conditions: k_switchback --> k_helix as s --> +- infinity (away from switchback point)
    # fixed_theta_indices = self.state.frozen_theta_indices
    # free_theta_indices = np.setdiff1d(np.arange(theta.size), fixed_theta_indices)
    # theta_fixed = theta[fixed_theta_indices]
    # n_edges = theta.size
    vec0 = params_to_vec(params)
    helix_copy = deepcopy(helix)

    # Insert the switchback — updates helix.q (k1, k2, k3)
    switchback_helix = switchback_k(helix_copy, params, point)

    # Treat this q as the rest configuration
    pos, theta = RodHelixConverter.helix_to_rod(switchback_helix)
    n_sites, n_edges = pos.shape[0], theta.shape[0]

    sim = Sim(
        pos=pos,
        theta=theta,
        frozen_pos_indices=np.array([0]),
        frozen_theta_indices=np.array([0]),
        B=B,  # defined as before
        beta=0.1,
        k=0.0,
        g=9.81 * 1e-3,
        mass=np.ones(pos.shape[0]),
        energies=[Gravity(), Twist(), Bend(), BendTwist()],
        damping=0.2,
        dt=0.04,
        xpbd_steps=10
    )

    sim.state = sim.init_state

    def total_energy(t: np.ndarray):
        energy_tot = 0.0
        # Somehow enforce boundary conditions?

        for energy in sim.energies:
            energy_tot += energy.compute_energy(pos=pos, theta=theta, rod_state=sim.state,
                                                init_rod_state=sim.init_state, rod_params=sim.rod_params)
        return energy_tot

    def total_grad_energy(t: np.ndarray):
        grad = np.zeros(len(vec0))
        # full_theta = np.zeros(n_edges)
        # full_theta[fixed_theta_indices] = theta_fixed
        # full_theta[free_theta_indices] = t
        for energy in sim.energies:
            energy.d_energy_d_theta(grad=grad, pos=pos, theta=theta, rod_state=sim.state,
                                    init_rod_state=sim.init_state, rod_params=sim.rod_params)
        return grad[vec0]

    # Minimize total energy wrst the free theta values
    res = minimize(total_energy, vec0, jac=total_grad_energy, method='L-BFGS-B')
    return res

# Apply optimal parameters
res = minimize_energy(init_params, helix, point)
best_params = vec_to_params(res.x)
optimized_helix = switchback_k(deepcopy(helix), best_params, point)
# Propagate the optimized helix
r, n = HelixUtil.propagate(optimized_helix)

plot_helix_plotly([(r, n)], frame_scale=0.2, show_frames=False, dot_pts=[21])

/var/folders/08/9rbxcpbs2rl1znd03wn5hz6m0000gn/T/ipykernel_20756/2641744434.py:55: RuntimeWarning:

invalid value encountered in divide

/var/folders/08/9rbxcpbs2rl1znd03wn5hz6m0000gn/T/ipykernel_20756/2641744434.py:56: RuntimeWarning:

invalid value encountered in divide



NameError: name 'B' is not defined

In [13]:
param_bounds = np.array([
    [0.0, 0.1],            # helix radius (m)
    [0.0, 100.0],            # helix wavenumber (radial oscillation) [1/m]
    [0.0, 100.0],           # twist wavenumber [1/m]
    [35e-6, 50e-6],        # strand thickness (minor axis radius) (m)
    [0.0, 2.2],            # ellipticity (major/minor)
    [0.5e9, 2.0e9],        # Young's modulus (Pa)
])

def compute_bending_tensor(n_edges, minor_radius, ellipticity, youngs_modulus):
    """
    Compute bending stiffness tensor B for a rod with elliptical cross-section.
    
    Parameters:
    - minor_radius: b (m)
    - ellipticity: a/b ratio
    - youngs_modulus: in Pa
    """
    b = np.asarray(minor_radius)
    a = ellipticity * b
    I_x = (np.pi / 4) * a * b**3
    I_y = (np.pi / 4) * a**3 * b
    EI_x = youngs_modulus * I_x
    EI_y = youngs_modulus * I_y
    B = np.zeros((n_edges, 2, 2))
    B[:, 0, 0] = EI_x
    B[:, 1, 1] = EI_y
    return B

def add_twist(pos, theta, twist_freq):
    """
    Add random twist along the rod at a given spatial frequency (wavenumber, in 1/m).
    """
    helix = RodHelixConverter.rod_to_helix(pos=pos, theta=theta)
    arc_length = np.sum(np.linalg.norm(np.diff(pos, axis=0), axis=1))
    total_twists = int(twist_freq * arc_length)

    n_segments = len(helix.q) // 3
    twist_indices = np.random.choice(
        np.arange(n_segments),
        size=total_twists,
        replace=(total_twists > n_segments)
    )
    q_indices = 3 * twist_indices
    helix.q[q_indices] += np.random.uniform(-np.pi, np.pi, size=total_twists)
    
    return RodHelixConverter.helix_to_rod(helix=helix)

def make_strand(helix_radius, freq, twist_freq,
                strand_thickness, ellipticity, youngs_modulus, n_edges, offset):
    """
    Generate a perturbed helical strand with specified physical properties.
    """
    arc_length = 1  # total arc length in meters
    pos, theta = RodGenerator.example_rod(n_edges, helix_radius, freq, arc_length)
    
    pos, theta = add_twist(pos, theta, twist_freq)

    # Apply randomized in-plane shifts
    pos[:, 0] += offset
    pos[:, 1] += offset
    
    # Normalize starting point in y and z
    pos[:, 1] -= pos[0, 1]
    pos[:, 2] -= pos[0, 2]

    n_sites, n_edges = pos.shape[0], theta.shape[0]
    mass = np.full(n_sites, 0.0002)  # kg, unit mass
    # mass = np.ones(n_sites)

    B = compute_bending_tensor(n_edges=n_edges, minor_radius=strand_thickness,
                                ellipticity=ellipticity, youngs_modulus=youngs_modulus)

    sim = Sim(
        pos=pos,
        theta=theta,
        B=B,
        beta=1,
        k=0.0,
        #g=9.81e-3,  # mm/s²
        g=9.81, # m/s²
        mass=mass,
        energies=[Gravity(), Bend(), Twist(), BendTwist()],
        damping=0.1,
        dt=0.1,
        xpbd_steps=10,
        frozen_pos_indices=np.array([0], dtype=int),
        frozen_theta_indices=np.array([], dtype=int)
    )

    return pos, theta, sim

def strands_to_one_objs(strands: np.ndarray, frame_idx: int, output_file: str = None, y_up: bool = True):
    output_file = f"output/sampling_scratch/obj_{frame_idx}.obj" if output_file is None else output_file
    Visualizer.clear_output_file(output_file)
    vertex_offset = 1
    for strand in strands:
        pos = strand[:, :3]
        vertex_offset = Visualizer.to_simple_obj(pos=pos, output_file=output_file, init_offset=vertex_offset, y_up=y_up)
    return

# poses.append(pos)
# thetas.append(theta)
# sims.append(sim)

In [16]:
from scipy.stats import qmc

# Define parameter ranges
param_bounds = np.array([
    [0.1, 4],         # radius (m)
    [0, 0.5],           # frequency (m^-1)
    [0.1, 0.5]           # twist frequency (m^-1)
    # [35e-6, 50e-6],   # strand thickness (m)
    # [0, 2.2],         # ellipticity (major/minor)
    # [0.5, 2],         # Young's modulus (GPa)
])

n_params = param_bounds.shape[0]
n_samples = 20

sampler = qmc.LatinHypercube(d=n_params)
lhs_sample = sampler.random(n=n_samples)

# Scale to your ranges
scaled_samples = qmc.scale(lhs_sample, param_bounds[:,0], param_bounds[:,1])

# rads, freqs, twist_freqs, strand_thickness, ellipticity, youngs_m = scaled_samples.T

# Simulate
poses = []
thetas = []
sims = []

for i, sample in enumerate(scaled_samples):
    r, f , tf = sample
    print(r, f)
    pos, theta, sim = make_strand(r, f, tf, 1, 1.0, 1.0, 100, offset = i * 10)
    # pos, theta = add_twist(pos, theta, 0)
    poses.append(pos)
    thetas.append(theta)
    sims.append(sim)

strands_to_one_objs(np.array(poses), 1)

0.33622501558770607 0.3046255273832473
1.229491637232669 0.2391148871026581
1.5016837250698158 0.38566829754086357
3.412306502289465 0.1084242696359701
0.27552725848560267 0.4253332643667188
0.6396567007025048 0.007549979199356479
1.015358380904222 0.14666574392137857
2.835155317652985 0.07755884455331293
0.8671975553700219 0.4920219747139001
3.9461171440678346 0.263442140511081
2.6680984599225828 0.15036636878206017
2.1991976602364045 0.06865957937252418
2.371215889780879 0.20650692840096654
1.825517631759408 0.03977440242783757
3.1511759139847464 0.28896464318476184
1.4430955921453934 0.1879861652579687
3.4970890896512943 0.33285118990853824
1.9482418076137982 0.4164742394294799
2.511368509754739 0.3578860061256839
3.7374040559033244 0.4725823652062493


In [17]:
# running simulation setting generated positions as rest positions
tracking_freq = 5
progress = tqdm(range(20 * tracking_freq))
for i in progress:
    for j in range(n_samples):
        pos, theta = sims[j].step(pos=poses[j], theta=thetas[j])
        poses[j] = pos
        thetas[j] = theta
    if i % tracking_freq == 0:
        progress.set_description(f"Frame {i // tracking_freq + 1}")
        strands_to_one_objs(np.array(poses), i // tracking_freq + 1)

Frame 15:  73%|███████▎  | 73/100 [00:55<00:20,  1.32it/s]


KeyboardInterrupt: 

In [165]:
# Controls

## Only varying twist freq.
# r, f, tf, st, e, ym = 3, 100, 0, 35e-6, 1.0, 1.0
# tf = np.linspace(0, 100, 10)

# poses = []
# thetas = []
# sims = []

# for i, t_freq in enumerate(tf):
#     pos, theta, sim = make_strand(r, f, t_freq, st, e, ym, 100, offset = i * 10)
#     pos, theta = add_twist(pos, theta, t_freq)
#     poses.append(pos)
#     thetas.append(theta)
#     sims.append(sim)

## Only varying ellipticity
# r, f, tf, st, ym = 3, 30, 0, 35e-6, 1.0
# e = np.linspace(0, 2.2, 10)

# poses = []
# thetas = []
# sims = []

# for el in e:
#     pos, theta, sim = make_strand(r, f, tf, st, el, ym, 100)
#     pos, theta = add_twist(pos, theta, tf)
#     poses.append(pos)
#     thetas.append(theta)
#     sims.append(sim)

## Only varying curl frequency and radius, twist frequency

In [175]:
def strands_to_one_objs(strands: np.ndarray, frame_idx: int, output_file: str = None, y_up: bool = True):
    output_file = f"output/sampling_scratch/obj_{frame_idx}.obj" if output_file is None else output_file
    Visualizer.clear_output_file(output_file)
    vertex_offset = 1
    for strand in strands:
        pos = strand[:, :3]
        vertex_offset = Visualizer.to_simple_obj(pos=pos, output_file=output_file, init_offset=vertex_offset, y_up=y_up)
    return

def step_wrapper(i, pos, theta, sim, n_steps=10):
    for _ in range(n_steps):
        pos, theta = sim.step(pos=pos, theta=theta)
    return i, pos, theta, sim

tracking_freq = 5
progress = tqdm(range(200 * tracking_freq))
for i in progress:
    for j in range(n_samples):
        pos, theta = sims[j].step(pos=poses[j], theta=thetas[j])
        poses[j] = pos
        thetas[j] = theta
    if i % tracking_freq == 0:
        progress.set_description(f"Frame {i // tracking_freq}")
        strands_to_one_objs(np.array(poses), i // tracking_freq)

Frame 9:   5%|▍         | 49/1000 [00:41<13:19,  1.19it/s]


KeyboardInterrupt: 